# 02 — Montando e treinando a CycleGAN (KDD, fase 4)

Neste notebook você monta a CycleGAN **peça por peça**: o bloco residual, o gerador, o discriminador e as três losses. No fim, roda um treino curto e vê as primeiras fotos ganhando pinceladas.

A explicação de cada peça está em `docs/guia/06-cyclegan-por-dentro.md`. Leia antes, ou deixe aberto ao lado.

## Como usar

- **Rodar uma célula:** clique nela e aperte `Shift + Enter`. Rode de cima para baixo, sem pular.
- **`___` (três sublinhados)** marca o lugar que você completa. Se rodar sem completar, aparece `NameError: name '___' is not defined`. É normal, só quer dizer que falta preencher.
- **Verificação:** depois de cada `TODO` há uma célula que mostra ✅ (certo) ou ❌ (ainda não).
- **✍️ Perguntas:** responda com suas palavras, mesmo que seja um palpite.
- **Gabarito:** o mesmo código, já completo, está em `src/models.py` e `src/train_cyclegan.py`. Tente sozinho primeiro e use o gabarito para conferir.

> **Antes de começar:** termine o notebook 01. Este notebook usa as imagens que ele já deixou prontas em `data/`.

## 0. Preparação

Esta célula já está pronta. Ela importa as bibliotecas, escolhe a GPU e carrega as imagens em 128x128.

In [ ]:
import sys
sys.path.insert(0, "..")  # permite importar o código da pasta src/

import time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from src.utils import get_device, load_domain, random_batch, set_seed, to_float, to_unit


def confere(condicao, mensagem):
    # Mostra ✅ se a condição for verdadeira, ❌ se ainda não.
    print(("✅ " if condicao else "❌ ") + mensagem)


def mostra(imagens, titulo=None, nrow=8):
    # Mostra um tensor (N, 3, H, W) em [-1, 1] como uma grade de imagens.
    from torchvision.utils import make_grid
    grade = make_grid(to_unit(imagens.detach().float().cpu()).clamp(0, 1), nrow=nrow, padding=2)
    plt.figure(figsize=(2 * nrow, 2 * (len(imagens) + nrow - 1) // nrow))
    plt.imshow(grade.permute(1, 2, 0))
    plt.axis("off")
    if titulo:
        plt.title(titulo)
    plt.show()


set_seed(42)
device = get_device()
torch.backends.cudnn.benchmark = True  # escolhe a convolução mais rápida para o tamanho das imagens

pinturas = load_domain("pintura", train=True)
fotos = load_domain("foto", train=True)
fotos_teste = to_float(load_domain("foto", train=False)[:8]).to(device)

print("Dispositivo:", device)
print("Pinturas:", tuple(pinturas.shape), "| fotos:", tuple(fotos.shape))

---
# Parte 1: as peças

## TODO 1: o bloco residual

O gerador transforma a foto usando **blocos residuais**. Em vez de calcular uma imagem nova do zero, cada bloco calcula só uma **correção** e soma ao que entrou:

```
saída = entrada + correção(entrada)
```

Por que isso ajuda: para manter a foto como está, basta a correção ser zero. Isso é fácil de aprender, e é exatamente o que queremos, mudar o estilo sem destruir a cena.

In [ ]:
class ResnetBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.correcao = nn.Sequential(
            nn.ReflectionPad2d(1),   # "espelha" a borda, para a convolução não criar uma moldura escura
            nn.Conv2d(ch, ch, 3),
            nn.InstanceNorm2d(ch),
            nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(ch, ch, 3),
            nn.InstanceNorm2d(ch),
        )

    def forward(self, x):
        # TODO 1: devolva a entrada somada à correção.
        # Dica: a correção é self.correcao(x)
        return ___

In [ ]:
bloco = ResnetBlock(8)
x = torch.randn(1, 8, 16, 16)
confere(bloco(x).shape == x.shape, "o bloco mantém o tamanho")

# Zerando os pesos das convoluções, a correção vira zero... e a saída deveria ser a própria entrada
for m in bloco.modules():
    if isinstance(m, nn.Conv2d):
        nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)
confere(torch.allclose(bloco(x), x), "correção zero => saída = entrada")

**Pergunta:** a segunda verificação mostra que, com a correção zerada, o bloco devolve a entrada intacta. Por que isso é útil para converter uma foto em pintura **sem perder a estátua**?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

## TODO 2: o gerador

O gerador tem três partes:

1. **Codificador:** reduz a imagem de 128x128 para 32x32, mas com muito mais canais (de 3 para 256). É um "resumo" do que tem na foto.
2. **6 blocos residuais:** transformam o resumo.
3. **Decodificador:** volta para 128x128 com 3 canais.

A última camada precisa produzir valores na mesma faixa das imagens reais, que estão em [-1, 1].

In [ ]:
class Generator(nn.Module):
    def __init__(self, ngf=64, n_blocks=6):
        super().__init__()
        camadas = [
            # Codificador
            nn.ReflectionPad2d(3), nn.Conv2d(3, ngf, 7), nn.InstanceNorm2d(ngf), nn.ReLU(True),
            nn.Conv2d(ngf, ngf * 2, 3, 2, 1), nn.InstanceNorm2d(ngf * 2), nn.ReLU(True),
            nn.Conv2d(ngf * 2, ngf * 4, 3, 2, 1), nn.InstanceNorm2d(ngf * 4), nn.ReLU(True),
        ]
        camadas += [ResnetBlock(ngf * 4) for _ in range(n_blocks)]
        camadas += [
            # Decodificador
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 3, 2, 1, output_padding=1), nn.InstanceNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 3, 2, 1, output_padding=1), nn.InstanceNorm2d(ngf), nn.ReLU(True),
            nn.ReflectionPad2d(3), nn.Conv2d(ngf, 3, 7),
            # TODO 2: qual função de ativação leva qualquer número para a faixa [-1, 1]?
            # Dica: é a mesma do gerador de toda GAN de imagens. Em PyTorch: nn.Algumacoisa()
            ___,
        ]
        self.net = nn.Sequential(*camadas)

    def forward(self, x):
        return self.net(x)

In [ ]:
from src.models import Generator as GeneratorGabarito

G_teste = Generator()
saida = G_teste(torch.randn(2, 3, 128, 128))
confere(saida.shape == (2, 3, 128, 128), "a saída tem o mesmo tamanho da entrada")
confere(saida.min() >= -1 and saida.max() <= 1, "a saída está em [-1, 1]")

n_params = sum(p.numel() for p in G_teste.parameters())
n_gabarito = sum(p.numel() for p in GeneratorGabarito().parameters())
print(f"Parâmetros: {n_params:,} (gabarito: {n_gabarito:,})")
confere(n_params == n_gabarito, "mesma arquitetura do src/models.py")

Veja o caminho da imagem dentro do gerador, camada por camada. A célula já está pronta.

In [ ]:
x = torch.randn(1, 3, 128, 128)
for camada in G_teste.net:
    x = camada(x)
    if not isinstance(camada, (nn.ReLU, nn.InstanceNorm2d, nn.ReflectionPad2d)):
        print(f"{camada.__class__.__name__:<16} -> {tuple(x.shape)}")

**Perguntas:**
1. Em que tamanho a imagem fica durante os 6 blocos residuais? Quantos canais ela tem nesse ponto?
2. A foto tem 3 canais (vermelho, verde e azul). O que você imagina que representam os 256 canais do meio?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

## TODO 3: o discriminador PatchGAN

Um discriminador comum dá **uma nota** para a imagem inteira. O PatchGAN dá uma **grade de notas**: cada nota julga um pedaço da imagem, de cerca de 70x70 pixels.

O estilo de um pintor está nas pinceladas, ou seja, nos detalhes locais. Por isso faz sentido julgar pedaço por pedaço.

Este já vem pronto. O `TODO` é entender a saída.

In [ ]:
from src.models import PatchDiscriminator

D_teste = PatchDiscriminator()
notas = D_teste(torch.randn(1, 3, 128, 128))
print("Formato da saída:", tuple(notas.shape))

# TODO 3: quantas notas (pedaços julgados) o discriminador devolve para UMA imagem?
# Dica: multiplique a altura pela largura da grade de notas (os dois últimos números do formato)
n_notas = ___
print("Notas por imagem:", n_notas)

In [ ]:
confere(n_notas == 196, "uma grade 14x14 = 196 notas")

**Pergunta:** uma imagem de 128x128 tem 196 notas, mas cada nota olha um pedaço de ~70x70 pixels. Os pedaços se sobrepõem? O que isso significa para o sinal que o gerador recebe?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

---
# Parte 2: as losses

A CycleGAN usa três losses. Aqui você escreve cada uma como uma função pequena e testa com casos em que já sabemos a resposta.

## TODO 4: a loss do discriminador (LSGAN)

O discriminador quer dar nota **1 para as reais** e **0 para as falsas**. A LSGAN mede o erro como a distância ao quadrado até esse alvo (`nn.MSELoss`, o *mean squared error*). A média das duas partes vem multiplicada por 0,5, como no artigo.

In [ ]:
mse = nn.MSELoss()
l1 = nn.L1Loss()


def loss_discriminador(pred_real, pred_fake):
    # TODO 4: erro das reais até 1 + erro das falsas até 0, vezes 0.5.
    # Dica: torch.ones_like(t) cria um tensor de 1s do mesmo tamanho de t; torch.zeros_like(t), de 0s.
    #       mse(previsto, alvo) calcula o erro quadrático médio.
    return 0.5 * (mse(pred_real, ___) + mse(pred_fake, ___))

In [ ]:
um, zero, meio = torch.ones(1, 1, 14, 14), torch.zeros(1, 1, 14, 14), torch.full((1, 1, 14, 14), 0.5)
confere(loss_discriminador(um, zero).item() == 0, "D perfeito: loss 0")
confere(abs(loss_discriminador(meio, meio).item() - 0.25) < 1e-6, "D chutando 0.5 para tudo: loss 0.25")
confere(loss_discriminador(zero, um).item() == 1, "D totalmente enganado: loss 1")

**Pergunta:** guarde o valor **0,25**. Nas curvas de treino, se a loss de um discriminador ficar perto de 0,25, o que isso quer dizer sobre ele? E se ela for para perto de 0?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

## TODO 5: a loss adversarial do gerador

O gerador quer o contrário: que o discriminador dê nota **1** (real) para as imagens **falsas** dele.

In [ ]:
def loss_adversarial(pred_fake):
    # TODO 5: erro entre a nota que o D deu para a falsa e o alvo "real" (1).
    return mse(pred_fake, ___)

In [ ]:
confere(loss_adversarial(um).item() == 0, "D enganado: o gerador não tem do que reclamar")
confere(loss_adversarial(zero).item() == 1, "D não foi enganado: loss 1")

## TODO 6: a *cycle loss*

A foto vira pintura e volta a ser foto, `F(G(foto))`, e o resultado tem que ser igual à foto original. O mesmo vale no sentido contrário: `G(F(pintura))` tem que voltar à pintura.

A diferença é medida pixel a pixel com `nn.L1Loss`: a média do valor absoluto das diferenças.

In [ ]:
def loss_cycle(G, F, foto, pintura):
    # TODO 6: some as duas idas e voltas.
    # Dica: l1(volta, original). A volta da foto é F(G(foto)). Qual é a volta da pintura?
    return l1(F(G(foto)), foto) + l1(___, pintura)

In [ ]:
foto_x, pint_x = torch.rand(1, 3, 8, 8), torch.rand(1, 3, 8, 8)
nada = nn.Identity()                  # um "gerador" que não muda nada
inverte = lambda t: -t                # um "gerador" que inverte as cores
confere(loss_cycle(nada, nada, foto_x, pint_x).item() == 0, "geradores que não mudam nada: ciclo perfeito")
confere(loss_cycle(inverte, inverte, foto_x, pint_x).item() == 0, "inverter duas vezes volta ao original")
confere(loss_cycle(nada, inverte, foto_x, pint_x).item() > 0, "ir e não voltar: loss > 0")

**Pergunta:** a segunda verificação mostra que a *cycle loss* fica zerada até com geradores que **invertem todas as cores**. Então ela sozinha não garante uma pintura bonita. Qual das outras losses impede o G de simplesmente inverter as cores?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

## TODO 7: a *identity loss*

Se o G (foto → pintura) recebe algo que **já é** uma pintura, ele deveria devolvê-la quase igual. Sem essa regra, o gerador tende a mexer nas cores da imagem toda.

In [ ]:
def loss_identity(G, F, foto, pintura):
    # TODO 7: G recebe uma pintura e deve devolvê-la igual; F recebe uma foto e deve devolvê-la igual.
    return l1(G(pintura), pintura) + l1(___, foto)

In [ ]:
confere(loss_identity(nada, nada, foto_x, pint_x).item() == 0, "geradores que não mudam nada: identity 0")
confere(loss_identity(inverte, inverte, foto_x, pint_x).item() > 0, "inverter as cores é punido")

---
# Parte 3: um treino curto

Agora as peças viram um treino. Cada passo tem duas metades:

1. **Geradores:** calculam as três losses e se ajustam para enganar os discriminadores sem perder o conteúdo.
2. **Discriminadores:** aprendem a separar as reais das falsas. As falsas vêm de um ***image pool***, uma mistura das imagens novas com 50 antigas, para o D não reagir só à última "moda" do gerador.

## TODO 8: a loss total do gerador

```
loss_G = adversarial  +  λ_cycle × cycle  +  λ_identity × identity
```

Os pesos são os do artigo: λ_cycle = 10 e λ_identity = 5.

In [ ]:
from src.train_cyclegan import ImagePool


def treino_curto(passos=300, lambda_cycle=10.0, lambda_identity=5.0, mostrar_a_cada=100, seed=42):
    set_seed(seed)
    G, F = Generator().to(device), Generator().to(device)                       # foto -> pintura, pintura -> foto
    D_pintura, D_foto = PatchDiscriminator().to(device), PatchDiscriminator().to(device)
    opt_g = torch.optim.Adam(list(G.parameters()) + list(F.parameters()), lr=2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(list(D_pintura.parameters()) + list(D_foto.parameters()), lr=2e-4, betas=(0.5, 0.999))
    pool_pintura, pool_foto = ImagePool(50), ImagePool(50)
    historico = []

    t0 = time.time()
    for passo in range(1, passos + 1):
        foto = random_batch(fotos, 1, device)
        pintura = random_batch(pinturas, 1, device)

        # --- 1. Geradores
        opt_g.zero_grad()
        fake_pintura, fake_foto = G(foto), F(pintura)
        adv = loss_adversarial(D_pintura(fake_pintura)) + loss_adversarial(D_foto(fake_foto))
        cyc = loss_cycle(G, F, foto, pintura)
        idt = loss_identity(G, F, foto, pintura)
        # TODO 8: monte a loss total com os pesos lambda_cycle e lambda_identity
        loss_g = adv + ___
        loss_g.backward()
        opt_g.step()

        # --- 2. Discriminadores
        opt_d.zero_grad()
        loss_d_pintura = loss_discriminador(D_pintura(pintura), D_pintura(pool_pintura.query(fake_pintura)))
        loss_d_foto = loss_discriminador(D_foto(foto), D_foto(pool_foto.query(fake_foto)))
        (loss_d_pintura + loss_d_foto).backward()
        opt_d.step()

        historico.append({"adv": adv.item(), "cycle": cyc.item(), "identity": idt.item(),
                          "d_pintura": loss_d_pintura.item(), "d_foto": loss_d_foto.item()})
        if passo % mostrar_a_cada == 0:
            print(f"passo {passo} | cycle {cyc.item():.3f} | D_pintura {loss_d_pintura.item():.3f} "
                  f"| {(time.time() - t0) / passo * 1000:.0f} ms por passo")
            with torch.no_grad():
                mostra(torch.cat([fotos_teste, G(fotos_teste)]), f"passo {passo}: fotos (em cima) e G(foto) (embaixo)")
    return G, F, historico

**Antes de rodar, escreva sua previsão:** depois de só 300 passos (o treino completo tem dezenas de milhares), as fotos vão parecer pinturas de Van Gogh? Vão continuar reconhecíveis?

✍️ **Sua previsão:** _dê dois cliques nesta célula para escrever_

A célula abaixo leva cerca de 1 minuto.

In [ ]:
G, F, historico = treino_curto(passos=300)

In [ ]:
confere(len(historico) == 300, "300 passos de treino")
confere(historico[-1]["cycle"] < historico[0]["cycle"], "a cycle loss caiu desde o primeiro passo")

Veja as curvas. Com uma imagem por passo, os valores oscilam muito, então a célula também desenha a média dos últimos 20 passos.

In [ ]:
import pandas as pd

h = pd.DataFrame(historico)
fig, eixos = plt.subplots(1, 2, figsize=(12, 3.5))
for col in ("adv", "cycle", "identity"):
    eixos[0].plot(h[col].rolling(20, min_periods=1).mean(), label=col)
eixos[0].set_title("geradores")
for col in ("d_pintura", "d_foto"):
    eixos[1].plot(h[col].rolling(20, min_periods=1).mean(), label=col)
eixos[1].axhline(0.25, ls="--", c="gray", label="D chutando (0.25)")
eixos[1].set_title("discriminadores")
for ax in eixos:
    ax.set_xlabel("passo")
    ax.legend()
plt.show()

**Perguntas:**
1. Sua previsão bateu? O que mudou nas fotos entre o passo 100 e o 300?
2. Qual loss caiu mais rápido? Por que você acha que essa é a mais fácil de aprender?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

## Experimento: e sem a *cycle loss*?

Essa é a comparação principal do projeto. O mesmo treino, mas com `lambda_cycle=0`.

**Antes de rodar, escreva sua previsão:** o que vai acontecer com o conteúdo das fotos?

✍️ **Sua previsão:** _dê dois cliques nesta célula para escrever_

In [ ]:
G_sem, F_sem, historico_sem = treino_curto(passos=300, lambda_cycle=0.0)

In [ ]:
with torch.no_grad():
    mostra(torch.cat([fotos_teste, G(fotos_teste), G_sem(fotos_teste)]),
           "fotos | G completa | G sem cycle loss")

**Perguntas:**
1. Sua previsão bateu? Em qual das duas versões as fotos ficaram mais reconhecíveis?
2. Com só 300 passos a diferença pode ser pequena. O que você espera ver no treino longo?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

> Registre as duas previsões e o que você observou em `docs/experimentos/` (peça ao Claude). Essas notas vão para o relatório.

---
# Parte 4: o treino longo, no terminal

O treino de verdade leva horas. No notebook ele trava se o computador dormir ou a aba fechar. Por isso ele roda no terminal, com o script `src/train_cyclegan.py`, que é **o mesmo código que você montou aqui**, com algumas coisas a mais:

- salva um *checkpoint* a cada 5 épocas e pode continuar de onde parou (`--resume`);
- diminui o *learning rate* aos poucos na segunda metade do treino;
- salva uma grade por época em `outputs/cyclegan/samples/` e as losses em `outputs/cyclegan/logs.csv`.

Com o venv ativado, na raiz do projeto:

```powershell
python -m src.train_cyclegan --epochs 1 --photos 300                  # teste: confira o tempo por passo
python -m src.train_cyclegan --epochs 40 --photos 2000                # treino completo
python -m src.train_cyclegan --epochs 40 --photos 2000 --sem-cycle    # a comparação
```

Cada época usa `--photos` passos. Com ~130 ms por passo, 2.000 fotos dão uns 4 minutos por época e cerca de 3 horas no total. Antes de decidir os números, leia a [ADR 0002](../docs/adr/0002-dataset-e-modelos.md) (seção "Em aberto").

Enquanto treina, abra as imagens de `outputs/cyclegan/samples/`: linha 1 = fotos, linha 2 = G(foto), linha 3 = a volta, F(G(foto)).

Depois do treino, o próximo notebook avalia os modelos e aplica o céu de Van Gogh nas suas fotos.